In [1]:
import os
import csv
import chromadb
from dotenv import load_dotenv
from google import genai
import pandas as pd
from chromadb import Documents, EmbeddingFunction, Embeddings
from IPython.display import Markdown

In [2]:
load_dotenv()
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

In [3]:
for m in client.models.list():
  if 'embedContent' in m.supported_actions:
    print(m.name)

models/embedding-001
models/text-embedding-004
models/gemini-embedding-exp-03-07
models/gemini-embedding-exp
models/gemini-embedding-001


In [4]:
# TODO: https://github.com/google-gemini/cookbook/blob/main/examples/chromadb/Vectordb_with_chroma.ipynb

In [5]:
documents = []
with open('output.csv', 'r', encoding='utf-8') as file:
    loaded_csv = csv.reader(file)
    header = next(loaded_csv)


    for row in loaded_csv:
        time_stamps = row[0]
        doc = row[1]+"\n"+"Segment Subtitle"+row[3]+"\n"+"Segment keywords:"+row[2].strip('[]')
        documents.append((time_stamps, doc))

In [6]:
documents[:5]

[('0s-10s',
  "This segment presents a stark visual of a completely black screen, punctuated by a slowly rotating, orange circle. The accompanying text introduces the fundamental concept of a vector in linear algebra, stating that it's the ‘root-of-it-all building block.’ The visual simplicity and the initial definition establish a foundational element for the video's subsequent explanation.\nSegment SubtitleThe fundamental, root-of-it-all building block for linear algebra is the vector.\nSegment keywords:'vector', 'linear algebra', 'black screen', 'circle', 'fundamentals'"),
 ('10s-20s',
  "The segment continues with a black screen and the phrase ‘The fundamental, root-of-it-all building block for linear algebra is the vector.’ This reinforces the initial concept.  The subsequent visuals show a vector represented as an arrow on a grid, highlighting its length and direction. The grid provides a visual context for understanding the vector's position in space.\nSegment SubtitleThe fundam

In [7]:
from google.genai import types

class GeminiEmbeddingFunction(EmbeddingFunction):
  def __call__(self, input: Documents) -> Embeddings:
    EMBEDDING_MODEL_ID = "models/embedding-001"  # @param ["models/embedding-001", "models/text-embedding-004", "models/gemini-embedding-exp-03-07", "models/gemini-embedding-exp"] {"allow-input": true, "isTemplate": true}
    title = "Custom query"
    response = client.models.embed_content(
        model=EMBEDDING_MODEL_ID,
        contents=input,
        config=types.EmbedContentConfig(
          task_type="retrieval_document",
          title=title
        )
    )

    return [emb.values for emb in response.embeddings]

In [8]:
def create_chroma_db(documents, name):
  chroma_client = chromadb.PersistentClient(path='./chromadb')
  db = chroma_client.get_or_create_collection(
      name=name,
      embedding_function=GeminiEmbeddingFunction()
  )

  for i, d in enumerate(documents):
    db.add(
      documents=d[1],
      ids=str(i),
      metadatas={'timestamps':d[0]}
    )
  return db

In [9]:
db = create_chroma_db(documents, "3b1b_vector")


/tmp/ipykernel_51999/108476745.py:5: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  embedding_function=GeminiEmbeddingFunction()


In [10]:
sample_data = db.get(include=['documents', 'embeddings'])

df = pd.DataFrame({
    "IDs": sample_data['ids'][:3],
    "Documents": sample_data['documents'][:3],
    "Embeddings": [str(emb)[:50] + "..." for emb in sample_data['embeddings'][:3]]  # Truncate embeddings
})

print(df)

  IDs                                          Documents  \
0   0  This segment presents a stark visual of a comp...   
1   1  The segment continues with a black screen and ...   
2   2  This segment introduces the three perspectives...   

                                          Embeddings  
0  [ 4.31862846e-02 -6.79173768e-02 -7.30723329e-...  
1  [ 4.86326823e-03 -4.40525115e-02  5.06896386e-...  
2  [-2.02238038e-02 -5.14030829e-02  5.28389867e-...  


In [ ]:
def get_relevant_passage(query, db):
  ret_seg_data = []
  retrieved_data = db.query(query_texts=[query], n_results=10)
  ret_ids = retrieved_data['ids'][0] 
  ret_data = db.get(ids=ret_ids)
  ts = ret_data['metadatas']
  docs = ret_data['documents']

  for i in range(10):
    chunk = {"time_stamps":ts[i]['timestamps'], "Segment content": docs[i] }
    ret_seg_data.append(chunk)
  

  
  return "".join(ret_seg_data)

In [ ]:
# Perform embedding search
passage = get_relevant_passage("vector addition", db)
passage

In [ ]:

def make_prompt(query, relevant_passage):
  prompt = f"""
    You are a helpful and informative video chunk/segment finder bot that find 
    relevant video chunk/segment description related to the QUESTION from the CHUNK LIST.
    Then output the relevant chunks with time_stamps in array like
    [20s-50s, 50s-120s, ...].
    If the chunk/segemnts are irrelevant then you may ignore it.
    QUESTION: {query}
    CHUNK LIST: {relevant_passage}

    ANSWER:
  """

  return prompt

In [ ]:
query = "Which sections are about vector addition?"
prompt = make_prompt(query, passage)
prompt

In [ ]:
MODEL_ID = "gemma-3-27b-it"  # @param ["gemini-2.5-flash-lite-preview-06-17", "gemini-2.5-flash", "gemini-2.5-flash","gemini-2.5-pro"] {"allow-input": true, "isTemplate": true}
answer = client.models.generate_content(
    model = MODEL_ID,
    contents = prompt
)
Markdown(answer.text)